# otwin 06 · The twin that says no

**Question.** You have a model, a validated forecast and a calibrated band. Someone asks it a
question — about a horizon it was never tested on, a state it never saw, a parameter it never
pinned down. What should it say?

**What otwin does here.** `TwinManifest` records what was done — which structure, which
parameters were estimated, how it was validated, how the band was calibrated, whether the
coefficients were identified. `Envelope` reads that record and either answers or refuses, with
a reason. `identifiability` produces the fourth verdict, the one notebook 03 detected by hand.

**What you write yourself.** Almost nothing. This is the smallest module in the library and
the one that decides whether anything upstream of it gets used.

**What you will learn**

1. what a manifest is, and why every key in it is strict (`True`, not truthy)
2. the four grounds for refusal — range, horizon, calibration, identification — and how each reads
3. how to check whether a fitted coefficient was determined by the data or chosen by the noise
4. why a refusal is an answer, and the one a lender should read first

*Pure simulation. Runtime < 1 min. Opens in Colab.*


## Setup

In [1]:
!pip install -q otwin 2>/dev/null
import otwin, sys
print("otwin", otwin.__version__, "| python", sys.version.split()[0])
from packaging.version import Version
assert Version(otwin.__version__) >= Version("0.4.0"), "this notebook needs otwin >= 0.4.0 for identifiability"
import numpy as np


otwin 0.4.0 | python 3.12.3


## 1 · The manifest: a record, not a description

A `TwinManifest` says what kind of model this is and what was done to it. Three of its fields
are built with helpers — `validated_by`, `calibrated_by`, `identified_by` — because each has
one key the envelope will read, and the helper puts it there.

The values below are the B0005 twin from notebook 03: fade law linear in throughput, one
estimated coefficient, rolling-origin validation, a split-conformal band that measured 87 %.

*Look at:* `is_validated` is `True` only because `leakage_free=True` is the boolean `True`.
`is_identified` is `False` — nothing was recorded yet. Not checked is not fine.

In [2]:
from otwin.interfaces import TwinManifest, Provenance
from otwin.advise import Envelope

prov = Provenance(created="2026-08-24T00:00:00Z", otwin_version=otwin.__version__,
                  data_source="NASA PCoE B0005 discharge", seed=0,
                  notes="fade law c*Q, residual GP, split at cycle 100 (notebook 03)")
twin = TwinManifest(
    name="b0005_fade", model_class="empirical_law", model_kind="battery_soh_fade",
    n_states=1, n_inputs=0, provenance=prov,
    parameters=dict(c=1.36e-3, k_thr=1.02, cap1=1.8565),
    estimated=("c",),
    validation=TwinManifest.validated_by("rolling_origin", leakage_free=True, theil_u=0.64, horizon=68),
    calibration=TwinManifest.calibrated_by("split_conformal", empirical_coverage=0.87, level=0.90),
)
print("white box?", twin.is_white_box, "| validated?", twin.is_validated,
      "| calibrated coverage:", twin.calibration["empirical_coverage"], "| identified?", twin.is_identified)


white box? False | validated? True | calibrated coverage: 0.87 | identified? False


## 2 · Three grounds for refusal

An `Envelope` holds the range the twin was identified over and the horizon it was validated
to. `check` compares a question against it and against the manifest. Four questions:

*Look at:* each refusal names the field it read and the two numbers it compared. The last one
is refused not because the answer would be wrong but because the band's coverage was never
measured — a request for an interval carries a heavier evidence requirement than a request for
a point.

In [3]:
env = Envelope(state_bounds=[(0.70, 1.00)], max_horizon=68,
               requires_validated=True, requires_calibrated=True)

for label, state, horizon in [("in range, in horizon",           [0.85], 40),
                               ("horizon far beyond validation",  [0.85], 900),
                               ("state below the validated range",[0.55], 40)]:
    v = env.check(state=state, horizon=horizon, manifest=twin, wants_interval=True)
    print("%-34s -> %s" % (label, "ANSWER" if v else "REFUSE"))
    for b in v.breaches: print("      ", b)

uncal = TwinManifest(name="b0005_uncal", model_class="empirical_law", model_kind="battery_soh_fade",
                     n_states=1, n_inputs=0, provenance=prov, parameters=dict(c=1.36e-3), estimated=("c",),
                     validation=TwinManifest.validated_by("rolling_origin", leakage_free=True, theil_u=0.64))
v = env.check(state=[0.85], horizon=40, manifest=uncal, wants_interval=True)
print("%-34s -> %s" % ("same question, no calibration", "ANSWER" if v else "REFUSE"))
for b in v.breaches: print("      ", b)


in range, in horizon               -> ANSWER
horizon far beyond validation      -> REFUSE
       horizon: beyond the validated forecast horizon (asked for 900, validated to 68)
state below the validated range    -> REFUSE
       state: state 0 below the identified range (asked for 0.55, validated to 0.7)
same question, no calibration      -> REFUSE
       calibration: interval coverage has never been measured, so the band has no demonstrated meaning


## 3 · The fourth ground: was the coefficient determined by the data?

Notebook 03 showed the free exponent `z` swinging from split to split. That is the symptom of
a parameter the data cannot pin down — *non-identifiability*. `otwin.estimate.identifiability`
tests it directly from the design matrix: can each column be reproduced from the others, is the
record longer than any time constant, and does the coefficient survive a bootstrap over units.

Here, a two-term fade law `c_slow·n^0.5 + c_knee·n^2.5` on the first 100 cycles of a synthetic
cell, and then on 800.

*Look at:* the same law, the same code. At 100 cycles `c_knee` is **not identified** — the
bootstrap returns a different value each time and switches the term off on some resamples.
At 800 it is. The report names which check failed, so you know whether to wait for data or
change the law: here, wait.

In [4]:
from otwin.estimate import identifiability

rng = np.random.default_rng(0)
n_all = np.arange(1.0, 801.0)
soh = 1 - 2.0e-3 * n_all**0.5 - 2.0e-9 * n_all**2.5 + rng.normal(0, 1.5e-3, n_all.size)

reports = {}
for window in (100, 800):
    n = n_all[:window]
    X = np.column_stack([n**0.5, n**2.5])
    reports[window] = identifiability(X, 1 - soh[:window], names=("c_slow", "c_knee"), nonneg=True, n_boot=200)
    print(f"{window} cycles -> {reports[window].verdicts}")
print()
print(reports[100].parameters[1])


100 cycles -> {'c_slow': True, 'c_knee': False}


800 cycles -> {'c_slow': True, 'c_knee': True}

c_knee: NOT identified (collinearity R²=0.758, bootstrap CV=0.72)
    bootstrap over 100 units gives CV=0.72 > 0.5; refitting on a resampled fleet returns a different value
    coefficient switches off or changes sign across bootstrap resamples


## 4 · Recording it, and refusing on it

The verdicts go into the manifest through `identified_by`, next to the other two records.
`Envelope(requires_identified=True)` then refuses a forecast that leans on an undetermined
coefficient — and names it.

*Look at:* the 100-cycle twin is inside its range, inside its horizon, validated and calibrated,
and is **refused anyway**, on the one ground the other three cannot see. The 800-cycle twin is
answered, and `checked` lists all four things that were examined.

In [5]:
def two_term_twin(report):
    return TwinManifest(
        name="cell-two-term", model_class="empirical_law", model_kind="two_term_fade",
        n_states=1, n_inputs=0, provenance=prov,
        parameters={"c_slow": 2.0e-3, "c_knee": 2.0e-9}, estimated=("c_slow", "c_knee"),
        validation=TwinManifest.validated_by("rolling_origin", leakage_free=True, horizon=60),
        calibration=TwinManifest.calibrated_by("horizon_conformal", empirical_coverage=0.90),
        identification=TwinManifest.identified_by("collinearity+bootstrap", parameters=report.verdicts),
    )

strict = Envelope(state_bounds=[(0.60, 1.00)], max_horizon=60, requires_identified=True)
for window in (100, 800):
    v = strict.check(state=[0.9], horizon=30, manifest=two_term_twin(reports[window]), wants_interval=True)
    print(f"fitted on {window} cycles -> {'ANSWER' if v else 'REFUSE'}")
    for b in v.breaches: print("      ", b)
    if v: print("       checked:", "; ".join(v.checked))


fitted on 100 cycles -> REFUSE
       identification: extrapolation depends on parameters the data did not determine: c_knee. A fitted value the fleet cannot pin down is a value chosen by the noise
fitted on 800 cycles -> ANSWER
       checked: horizon 30 <= 60; operating point inside the identified range; validated, leakage-free; estimated parameters identified; coverage measured at 0.90


## 5 · Change one thing

- **Loosen the envelope.** `max_extrapolation=0.05` lets the state go 5 % outside the identified
  range before refusing. That is a judgement about your asset; the library will not make it for you.
- **Write the manifest by hand.** Replace `validated_by(...)` with `dict(protocol="rolling_origin",
  picp=0.87)`. It is refused — the key the envelope reads is not there. The helpers exist so that
  this does not happen silently.
- **Ask for a point, not a band.** `wants_interval=False` on the uncalibrated twin: answered.
  Different question, different evidence requirement.

In [6]:
# Try it: a hand-written record with the wrong keys.
sloppy = TwinManifest(name="sloppy", model_class="empirical_law", model_kind="battery_soh_fade",
                      n_states=1, n_inputs=0, provenance=prov, parameters=dict(c=1.36e-3), estimated=("c",),
                      validation=dict(protocol="rolling_origin", picp=0.87),   # looks fine, reads as nothing
                      calibration=dict(method="split_conformal", picp=0.87))
v = env.check(state=[0.85], horizon=40, manifest=sloppy, wants_interval=True)
print("hand-written manifest ->", "ANSWER" if v else "REFUSE")
for b in v.breaches: print("      ", b)


hand-written manifest -> REFUSE
       validation: validation is recorded (picp, protocol) but does not assert leakage_free=True; that key, as the boolean, is what certifies the protocol. Build it with TwinManifest.validated_by(...)
       calibration: calibration is recorded (method, picp) but carries no empirical_coverage; a nominal level is a promise, and this check reads the measurement. Build it with TwinManifest.calibrated_by(...)


## What you learned

- A manifest is a strict record: the envelope reads specific keys, and the helpers set them.
- Four grounds for refusal — range, horizon, calibration, identification — each named in the verdict.
- A fitted coefficient is not a determined one; `identifiability` tells you which, and why.
- A refusal is an answer. It is the part of the report a careful reader looks at first.

**Next:** [notebook 07](otwin_07_field_data.ipynb) — all of this on eight years of real field
data: eighteen home-storage systems, sixty manual capacity tests, and a band whose coverage is
measured across systems it never saw.

---
*For CI, not for you.*

In [7]:
_checks = {
    "validated manifest answers in range": bool(env.check(state=[0.85], horizon=40, manifest=twin, wants_interval=True)),
    "900-step horizon refused":            not env.check(state=[0.85], horizon=900, manifest=twin),
    "uncalibrated band refused":           not env.check(state=[0.85], horizon=40, manifest=uncal, wants_interval=True),
    "c_knee not identified at 100 cycles": reports[100].verdicts["c_knee"] is False,
    "c_knee identified at 800 cycles":     reports[800].verdicts["c_knee"] is True,
    "unidentified twin refused":           not strict.check(state=[0.9], horizon=30, manifest=two_term_twin(reports[100])),
    "identified twin answered":            bool(strict.check(state=[0.9], horizon=30, manifest=two_term_twin(reports[800]))),
}
for k, ok in _checks.items(): print(f"{k:<40} {'yes' if ok else 'NO'}")
assert all(_checks.values()), [k for k, ok in _checks.items() if not ok]
print("\nregression test passed")


validated manifest answers in range      yes
900-step horizon refused                 yes
uncalibrated band refused                yes
c_knee not identified at 100 cycles      yes
c_knee identified at 800 cycles          yes
unidentified twin refused                yes
identified twin answered                 yes

regression test passed
